# 5단계 — Bumps ↔ Other_Faults 전용 보조 분류기 검증

**목표:** 기존 ExtraTrees + LightGBM 스태킹을 유지하면서, 가장 많이 발생하는 Bumps/Other_Faults 혼동만 선별적으로 재판정하여 Macro F1을 개선할 수 있는지 검증합니다.

- 기준: 개발 데이터(1,552행) 외부 3-Fold Macro F1 약 **0.8277** (seed=2027)
- 사전 탐색에서 유망했던 후보: Bumps/Other_Faults 전용 ExtraTrees, 신뢰도 `0.58`, 원래 두 클래스의 예측 확률 차이 `≤ 0.30`
- **새 분할(seed=2041) 재검증**으로 후보 선택에 따른 낙관적 편향을 확인합니다.
- 20% 홀드아웃(389행)은 학습·탐색·재검증에서 사용하지 않습니다. 목표 0.835가 달성된 것으로 선언하려면 최종 보류 데이터에서 한 번 더 평가해야 합니다.

**필요한 파일:** `Faults.NNA`, `Faults27x7_var` (이 노트북과 같은 폴더)

이 파일은 독립 실행형입니다. 1~4단계 노트북을 다시 실행할 필요가 없습니다.

In [1]:
# 필수 패키지: 미설치 시 아래 주석 해제 후 실행
# %pip install lightgbm scikit-learn pandas numpy

from pathlib import Path
import warnings
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore', category=FutureWarning)
root = Path('.')
for fname in ['Faults.NNA', 'Faults27x7_var']:
    if not (root/fname).exists():
        raise FileNotFoundError(f'{fname} 파일을 이 노트북과 같은 폴더에 넣어 주세요.')
cols = [line.strip() for line in (root/'Faults27x7_var').read_text(encoding='utf-8').splitlines() if line.strip()]
raw = pd.read_csv(root/'Faults.NNA', sep=r'\s+', header=None, names=cols)
assert raw.shape == (1941, 34), f'데이터 형식 확인 필요: {raw.shape}'
encoder = LabelEncoder()
y_full = encoder.fit_transform(raw[cols[27:]].idxmax(axis=1))
X_full = raw.iloc[:, :27].drop(columns='TypeOfSteel_A300')
idx_dev, idx_test = train_test_split(
    np.arange(len(y_full)), test_size=0.2, stratify=y_full, random_state=42
)
X_dev = X_full.iloc[idx_dev].reset_index(drop=True)
y_dev = y_full[idx_dev]
B = int(encoder.transform(['Bumps'])[0])
O = int(encoder.transform(['Other_Faults'])[0])
print('개발 데이터:', X_dev.shape, '| 사용하지 않는 홀드아웃:', len(idx_test))
print('클래스:', list(encoder.classes_))

개발 데이터: (1552, 26) | 사용하지 않는 홀드아웃: 389
클래스: ['Bumps', 'Dirtiness', 'K_Scatch', 'Other_Faults', 'Pastry', 'Stains', 'Z_Scratch']


In [2]:
# 기존 3단계와 동일한 ET/LGB 스태킹 설정
et_base = ExtraTreesClassifier(
    random_state=42, n_jobs=1, n_estimators=500,
    min_samples_split=5, min_samples_leaf=1,
    max_features=0.8, max_depth=None,
    class_weight='balanced_subsample'
)
lgb_base = LGBMClassifier(
    objective='multiclass', random_state=42, n_jobs=1, verbose=-1,
    subsample_freq=1, subsample=0.8, reg_lambda=0,
    num_leaves=22, n_estimators=700, min_child_samples=5,
    max_depth=-1, learning_rate=0.05,
    colsample_bytree=0.6, class_weight=None
)

# 클래스 혼동을 줄이기 위해 두 클래스 자료만으로 학습하는 보조 모델
pair_et = ExtraTreesClassifier(
    n_estimators=450, min_samples_leaf=2, max_features=0.9,
    random_state=42, n_jobs=1, class_weight='balanced'
)

# 2027 검증 분할에서 미리 선택한 한 가지 후보를 고정
SPECIALIST_CONFIDENCE = 0.58
MAX_PAIR_PROB_GAP = 0.30
MIN_PAIR_PROB_SUM = 0.50


def macro(y_true, y_pred):
    return f1_score(y_true, y_pred, average='macro', zero_division=0)


def evaluate(seed):
    outer = StratifiedKFold(n_splits=3, shuffle=True, random_state=seed)
    inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)
    rows = []
    y_all, base_all, revised_all = [], [], []

    for fold, (tr, va) in enumerate(outer.split(X_dev, y_dev), 1):
        Xa, Xv = X_dev.iloc[tr], X_dev.iloc[va]
        ya, yv = y_dev[tr], y_dev[va]

        # 외부 학습 Fold에서만 OOF 확률 생성(스태킹 누수 방지)
        et_oof = cross_val_predict(
            clone(et_base), Xa, ya, cv=inner,
            method='predict_proba', n_jobs=1
        )
        lgb_oof = cross_val_predict(
            clone(lgb_base), Xa, ya, cv=inner,
            method='predict_proba', n_jobs=1
        )
        meta = LogisticRegression(max_iter=3000, C=1.0).fit(
            np.hstack([et_oof, lgb_oof]), ya
        )
        et_valid = clone(et_base).fit(Xa, ya).predict_proba(Xv)
        lgb_valid = clone(lgb_base).fit(Xa, ya).predict_proba(Xv)
        meta_valid = np.hstack([et_valid, lgb_valid])
        base_pred = meta.predict(meta_valid)
        base_prob = meta.predict_proba(meta_valid)

        # 외부 Fold의 학습 부분 중 Bumps / Other_Faults 사례만 사용
        mask_train = np.isin(ya, [B, O])
        pair_target = (ya[mask_train] == B).astype(int)
        specialist = clone(pair_et).fit(Xa.loc[mask_train], pair_target)
        prob_bumps = specialist.predict_proba(Xv)[:, 1]

        # 메인 모델이 Bumps 또는 Other_Faults로 분류했고,
        # 두 클래스에 대한 상대적 불확실성이 존재하는 경우만 후보로 지정
        eligible = (
            np.isin(base_pred, [B, O])
            & ((base_prob[:, B] + base_prob[:, O]) >= MIN_PAIR_PROB_SUM)
            & (np.abs(base_prob[:, B] - base_prob[:, O]) <= MAX_PAIR_PROB_GAP)
            & ((prob_bumps >= SPECIALIST_CONFIDENCE)
               | (prob_bumps <= 1 - SPECIALIST_CONFIDENCE))
        )
        revised = base_pred.copy()
        revised[eligible] = np.where(prob_bumps[eligible] >= 0.5, B, O)

        changed = (revised != base_pred)
        corrected = int(np.sum(changed & (revised == yv) & (base_pred != yv)))
        worsened = int(np.sum(changed & (base_pred == yv) & (revised != yv)))

        rows.append({
            'Fold': fold, '기존 스태킹': macro(yv, base_pred),
            '보조 분류기': macro(yv, revised),
            '변경 건수': int(changed.sum()),
            '정답으로 개선': corrected, '정답에서 오답': worsened
        })
        y_all.append(np.asarray(yv))
        base_all.append(base_pred)
        revised_all.append(revised)
        print(
            f'분할 {seed} Fold {fold} | 기존={rows[-1]["기존 스태킹"]:.4f}'
            f' | 개선후={rows[-1]["보조 분류기"]:.4f}'
            f' | 변경={int(changed.sum())}', flush=True
        )
    result = pd.DataFrame(rows)
    return result, np.concatenate(y_all), np.concatenate(base_all), np.concatenate(revised_all)

print('보조 분류기 설정 준비 완료')

보조 분류기 설정 준비 완료


In [3]:
# [5-1] 기존 탐색 분할(seed=2027) 재현
explore, y_ex, pred_ex, revised_ex = evaluate(seed=2027)
explore['차이'] = explore['보조 분류기'] - explore['기존 스태킹']
print('\n[탐색 분할 결과]')
display(explore.round(4))
print('평균 Macro F1:', explore[['기존 스태킹','보조 분류기']].mean().round(4).to_dict())
explore.to_csv('stage5_explore_scores.csv', index=False, encoding='utf-8-sig')

분할 2027 Fold 1 | 기존=0.8238 | 개선후=0.8336 | 변경=14
분할 2027 Fold 2 | 기존=0.8193 | 개선후=0.8204 | 변경=11
분할 2027 Fold 3 | 기존=0.8401 | 개선후=0.8446 | 변경=7

[탐색 분할 결과]


,Fold,기존 스태킹,보조 분류기,변경 건수,정답으로 개선,정답에서 오답,차이
0,1,0.8238,0.8336,14,10,2,0.0098
1,2,0.8193,0.8204,11,5,4,0.0012
2,3,0.8401,0.8446,7,5,1,0.0044


평균 Macro F1: {'기존 스태킹': 0.8277, '보조 분류기': 0.8329}


In [4]:
# [5-2] 보조 분류기와 변경 조건을 고정하고 새 분할(seed=2041)로 재검증
verify, y_v, pred_v, revised_v = evaluate(seed=2041)
verify['차이'] = verify['보조 분류기'] - verify['기존 스태킹']
print('\n[새로운 분할 재검증]')
display(verify.round(4))
summary = pd.DataFrame({
    '평균 Macro F1': verify[['기존 스태킹','보조 분류기']].mean(),
    '표준편차': verify[['기존 스태킹','보조 분류기']].std(ddof=0)
})
display(summary.round(4))
gain = summary.loc['보조 분류기', '평균 Macro F1'] - summary.loc['기존 스태킹', '평균 Macro F1']
print(f'새 분할 평균 개선량: {gain:+.4f}')
print('목표 0.835 달성 여부:', '재검증 분할에서 목표 도달' if summary.loc['보조 분류기','평균 Macro F1'] >= 0.835 else '재검증 분할에서 목표 미달')
print('※ 개발 데이터의 다른 CV 분할일 뿐, 완전히 독립된 테스트 데이터는 아닙니다.')
verify.to_csv('stage5_verify_scores.csv', index=False, encoding='utf-8-sig')

분할 2041 Fold 1 | 기존=0.8256 | 개선후=0.8278 | 변경=2
분할 2041 Fold 2 | 기존=0.8078 | 개선후=0.8088 | 변경=10
분할 2041 Fold 3 | 기존=0.8159 | 개선후=0.8191 | 변경=7

[새로운 분할 재검증]


,Fold,기존 스태킹,보조 분류기,변경 건수,정답으로 개선,정답에서 오답,차이
0,1,0.8256,0.8278,2,2,0,0.0022
1,2,0.8078,0.8088,10,5,4,0.0010
2,3,0.8159,0.8191,7,4,1,0.0032


,평균 Macro F1,표준편차
기존 스태킹,0.8165,0.0073
보조 분류기,0.8186,0.0078


새 분할 평균 개선량: +0.0021
목표 0.835 달성 여부: 재검증 분할에서 목표 미달
※ 개발 데이터의 다른 CV 분할일 뿐, 완전히 독립된 테스트 데이터는 아닙니다.


In [5]:
# [5-3] 개선이 어느 클래스에 일어났는지 확인
names = list(encoder.classes_)
print('[원래 스태킹: 새 분할]')
print(classification_report(y_v, pred_v, labels=np.arange(len(names)), target_names=names, digits=4, zero_division=0))
print('[보조 분류기 적용: 새 분할]')
print(classification_report(y_v, revised_v, labels=np.arange(len(names)), target_names=names, digits=4, zero_division=0))
print('[보조 적용 후 오분류 행렬]')
display(pd.DataFrame(confusion_matrix(y_v, revised_v), index=names, columns=names))
print('※ 테스트셋은 사용하지 않았습니다. 두 모델의 성능 차이가 재현되는지 확인한 후에만 최종 홀드아웃 평가를 진행하세요.')

[원래 스태킹: 새 분할]
              precision    recall  f1-score   support

       Bumps     0.6948    0.6667    0.6804       321
   Dirtiness     0.9062    0.6591    0.7632        44
    K_Scatch     0.9869    0.9617    0.9741       313
Other_Faults     0.7085    0.7862    0.7454       538
      Pastry     0.7453    0.6270    0.6810       126
      Stains     0.9483    0.9483    0.9483        58
   Z_Scratch     0.9452    0.9079    0.9262       152

    accuracy                         0.7983      1552
   macro avg     0.8479    0.7938    0.8169      1552
weighted avg     0.8026    0.7983    0.7986      1552

[보조 분류기 적용: 새 분할]
              precision    recall  f1-score   support

       Bumps     0.6940    0.6854    0.6897       321
   Dirtiness     0.9062    0.6591    0.7632        44
    K_Scatch     0.9869    0.9617    0.9741       313
Other_Faults     0.7194    0.7862    0.7513       538
      Pastry     0.7453    0.6270    0.6810       126
      Stains     0.9483    0.9483    0.9483  

,Bumps,Dirtiness,K_Scatch,Other_Faults,Pastry,Stains,Z_Scratch
Bumps,220,1,0,91,8,1,0
Dirtiness,2,29,0,11,2,0,0
K_Scatch,1,0,301,11,0,0,0
Other_Faults,82,2,4,423,17,2,8
Pastry,12,0,0,35,79,0,0
Stains,0,0,0,3,0,55,0
Z_Scratch,0,0,0,14,0,0,138


※ 테스트셋은 사용하지 않았습니다. 두 모델의 성능 차이가 재현되는지 확인한 후에만 최종 홀드아웃 평가를 진행하세요.
